# Predicting Irrigation Need - GPU Notebook Workflow

This notebook is designed for **Kaggle Notebook** execution and creates a valid submission file (`submission_ready.csv`) with 270000 rows.

## 1) Install dependency (if needed)

In [ ]:
!pip -q install catboost

## 2) Imports + config

Adjust only the values in `CONFIG` for faster/slower runs.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, train_test_split

def detect_data_dir() -> Path:
    root = Path('/kaggle/input')
    if not root.exists():
        raise FileNotFoundError('/kaggle/input does not exist in this runtime.')

    # 1) fast path: known competition dataset names
    candidates = [
        root / 'playground-series-s6e4',
        root / 'playground-series-s6e4-predicting-irrigation-need',
    ]
    for p in candidates:
        if (p / 'train.csv').exists() and (p / 'test.csv').exists():
            return p

    # 2) scan top-level datasets
    for p in root.iterdir():
        if p.is_dir() and (p / 'train.csv').exists() and (p / 'test.csv').exists():
            return p

    # 3) scan one level deeper (some datasets are nested)
    for p in root.iterdir():
        if not p.is_dir():
            continue
        for q in p.iterdir():
            if q.is_dir() and (q / 'train.csv').exists() and (q / 'test.csv').exists():
                return q

    available = sorted([x.name for x in root.iterdir()])
    raise FileNotFoundError(
        'Could not find train.csv/test.csv under /kaggle/input. '
        f'Available datasets: {available}. '
        'Click Add Data in the right panel and add the competition dataset, then rerun.'
    )

CONFIG = {
    "data_dir": detect_data_dir(),
    "output_csv": Path("/kaggle/working/submission_ready.csv"),
    "output_report": Path("/kaggle/working/model_report.json"),
    "cv_sample_size": 220000,
    "cv_folds": 3,
    "cat_iterations": 700,
    "cat_seeds": [42, 314, 2024, 777],
    "gpu_devices": "0",
}
print('Detected data_dir:', CONFIG['data_dir'])

CONFIG

In [ ]:
def to_catboost_features(df: pd.DataFrame, categorical_features):
    out = df.copy()
    for c in categorical_features:
        out[c] = out[c].fillna("__MISSING__").astype(str)
    return out

def build_catboost_model(iterations: int, seed: int, gpu_devices: str):
    return CatBoostClassifier(
        iterations=iterations,
        depth=8,
        learning_rate=0.05,
        loss_function="MultiClass",
        auto_class_weights="Balanced",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        task_type="GPU",
        devices=gpu_devices,
    )

def aligned_proba_df(model, X, all_classes):
    proba = model.predict_proba(X)
    class_to_idx = {c: i for i, c in enumerate(model.classes_)}
    arr = np.zeros((len(X), len(all_classes)), dtype=float)
    for j, cls in enumerate(all_classes):
        arr[:, j] = proba[:, class_to_idx[cls]]
    return arr

def get_cv_subset(X, y, max_rows):
    if max_rows <= 0 or len(X) <= max_rows:
        return X, y
    X_sub, _, y_sub, _ = train_test_split(
        X, y, train_size=max_rows, random_state=42, stratify=y
    )
    return X_sub, y_sub

## 3) Load data and run CV (quick leaderboard signal)

In [ ]:
train_df = pd.read_csv(CONFIG["data_dir"] / "train.csv")
test_df = pd.read_csv(CONFIG["data_dir"] / "test.csv")
sample_sub = pd.read_csv(CONFIG["data_dir"] / "sample_submission.csv")

target_col = "Irrigation_Need"
id_col = "id"

y = train_df[target_col]
X = train_df.drop(columns=[target_col])
feature_cols = [c for c in X.columns if c != id_col]
X = X[feature_cols]
X_test = test_df[feature_cols]

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]
cat_features_idx = [X.columns.get_loc(c) for c in categorical_features]

X_cv, y_cv = get_cv_subset(X, y, CONFIG["cv_sample_size"])
X_cv_cb = to_catboost_features(X_cv, categorical_features)

cv = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=42)
cv_scores = []
for fold_idx, (tr_idx, va_idx) in enumerate(cv.split(X_cv_cb, y_cv), start=1):
    X_tr, X_va = X_cv_cb.iloc[tr_idx], X_cv_cb.iloc[va_idx]
    y_tr, y_va = y_cv.iloc[tr_idx], y_cv.iloc[va_idx]
    model = build_catboost_model(CONFIG["cat_iterations"], CONFIG["cat_seeds"][0], CONFIG["gpu_devices"])
    model.fit(X_tr, y_tr, cat_features=cat_features_idx)
    pred = model.predict(X_va).ravel()
    score = balanced_accuracy_score(y_va, pred)
    cv_scores.append(score)
    print(f"Fold {fold_idx} balanced accuracy: {score:.6f}")

print(f"CV mean: {np.mean(cv_scores):.6f} +/- {np.std(cv_scores):.6f}")

## 4) Train full ensemble and generate submission

In [ ]:
X_full_cb = to_catboost_features(X, categorical_features)
X_test_cb = to_catboost_features(X_test, categorical_features)
classes = sorted(y.unique())

seed_probas = []
for seed in CONFIG["cat_seeds"]:
    print(f"Training seed {seed}...")
    model = build_catboost_model(CONFIG["cat_iterations"], seed, CONFIG["gpu_devices"])
    model.fit(X_full_cb, y, cat_features=cat_features_idx)
    seed_probas.append(aligned_proba_df(model, X_test_cb, classes))

proba = np.mean(seed_probas, axis=0)
pred = np.array(classes)[proba.argmax(axis=1)]

submission = pd.DataFrame({id_col: test_df[id_col], target_col: pred})
submission.to_csv(CONFIG["output_csv"], index=False)
print(f"Saved: {CONFIG['output_csv']}")

## 5) Validate required shape + save run report

In [ ]:
out = pd.read_csv(CONFIG["output_csv"])
print("Output rows:", len(out))
print("Output columns:", list(out.columns))
print("Matches sample row count:", len(out) == len(sample_sub))
print("Matches sample headers:", list(out.columns) == list(sample_sub.columns))

report = {
    "cv_rows_used": int(len(X_cv)),
    "cv_folds": int(CONFIG["cv_folds"]),
    "cat_iterations": int(CONFIG["cat_iterations"]),
    "cat_seeds": CONFIG["cat_seeds"],
    "catboost_cv_mean": float(np.mean(cv_scores)),
    "catboost_cv_std": float(np.std(cv_scores)),
    "output_file": str(CONFIG["output_csv"]),
}
CONFIG["output_report"].write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Saved report: {CONFIG['output_report']}")

out.head()

## 6) (Optional) Create zip for competition upload widget

In [ ]:
!cd /kaggle/working && zip -q submission_ready.zip submission_ready.csv model_report.json && ls -lh /kaggle/working/submission_ready* /kaggle/working/model_report.json

### Suggested submission description

`GPU CatBoost multi-seed ensemble (4 seeds), balanced class weights, 3-fold CV check, full-train inference.`